# Evaluating Fantasy Football Player & Team Statistics

## Contents

| # | Section | Target Insight | Key Findings |
|---|---------|----------------|--------------|
| 1 | [Data & Setup](#1-data--setup) | What data, scoring format and features are we working with? | |
| 2 | [Drivers of Fantasy Performance](#drivers-of-fantasy-performance) | Which statistics correlate most strongly with fantasy points? | [The Most Important Statistics by Scoring Format](#the-most-important-statistics-by-scoring-format) |
| 3 | [Position-Specific Drivers](#3-position-specific-drivers) | How do the different positions relate to fantasy football performance? | [Positional Importance Differs by Scoring Format](#positional-importance-differs-by-scoring-format) |
| 4 | [Positional Draft Strategy](#4-positional-draft-strategy) | How should I prioritize selecting different positions? | [Ranking Positions by Draft Priority](#ranking-positions-by-draft-priority) |
| 5 | [Player Selection by Position](#5-player-selection-by-position) | How do the statistics relate to each other, and which ones identify strong players at each position? | [Selecting QBs](#selecting-qbs)<br>[Selecting RBs](#selecting-rbs)<br>[Selecting WRs](#selecting-wrs)<br>[Selecting TEs](#selecting-tes) |
| 6 | [Key Findings Summary & Recommendations](#7-key-findings-summary--recommendations) | What matters most, what's reliable, and what are the limitations? | |

In [ ]:
import sys
import tomllib
from pathlib import Path

In [ ]:
#CONFIG_PATH = Path("public-fantast-football-analytics/config/config.toml")
CONFIG_PATH = Path('config/config.toml')

with CONFIG_PATH.open("rb") as f:
    config = tomllib.load(f)

from dotenv import load_dotenv
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sys.path.append("public-fantast-football-analytics")

from src.data_processing import utils_data_cleansing as utils
from src.data_viz import viz
from src.data_processing import stats as stats


alt.renderers.enable("html")

RendererRegistry.enable('html')

## Introduction

## Data & Setup

# Load Data 

The grain of the data is two fold, NFL data is one row per player per week, while the college data is a season aggreate. As a result, we will need to primarily focus on returning NFL players when evualting efficency, which we will represent in points per week. 

In [ ]:
# Load the combined player stats CSV file into a DataFrame
player_stats_df = pd.read_csv("data/combined_player_stats.csv")
print("Column Count:", len(player_stats_df.columns))
print("Row Count:", len(player_stats_df))
display(player_stats_df.head(5))

# Split College and NFL stats into separate DataFrames
college_stats_df = player_stats_df[player_stats_df['data_source'] == 'college_data']
nfl_stats_df = player_stats_df[player_stats_df['data_source'] == 'nflreadpy']

Column Count: 162
Row Count: 165606


,player_id,player_name,player,position,position_group,headshot_url,season,week,season_type,game_id,...,playerId,conference,completion_pct,fg_points,pat_attempts,rec_long,rushing_long,yards_per_attempt,yards_per_carry,yards_per_rec
0,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2018,1.0,REG,2018_01_WAS_ARI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0016919,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2018,1.0,REG,2018_01_CIN_IND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2018,1.0,REG,2018_01_HOU_NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00-0019646,S.Janikowski,Sebastian Janikowski,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2018,1.0,REG,2018_01_SEA_DEN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00-0020531,D.Brees,Drew Brees,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2018,1.0,REG,2018_01_TB_NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# full PPR, all positions
all_ppr = stats.build_offense_season_agg(nfl_stats_df, groupby_cols=stats.all_player_cols, ppr=1, positions=stats.offensive_positions)
all_nonppr = stats.build_offense_season_agg(nfl_stats_df, groupby_cols=stats.all_player_cols, ppr=0, positions=stats.offensive_positions)

## Drivers of Fantasy Performance

This section looks at which variables correlate most strongly with fantasy points. 

Due to the limitations of the college data, this will be focused on returning NFL Players. 

- Players are intentionally not split by position here: the goal is to maximize points per week, not just to find position-specific patterns. 
- Pooling all positions shows which kinds of offensive production generate points most efficiently, and so which positions are the most efficient scorers overall.


**Definition:** Scoring Type
- PPR (points per reception): players earn 1 point for every catch, on top of yardage and touchdown points.
- Non-PPR (standard): catches earn no points on their own; only yardage and touchdowns count.

Throughout this analysis we wil be examining the data in both contexts as this single rule cand drastically affect how the game is played optimally. 

### Correlations 

The following visuals highlight moderate and strong correlations between statistics and player performance. 

In this first pass **we have not separated by posistion intentionally**. Before examining posistion specific performance drivers, we want to uncover high level trends which will provide the foundational strategy for selecting players. 

In [ ]:
ppr_corr_bar_chart = viz.corr_bar_chart(all_ppr, 'avg_fantasy_points', 'Correlation with Avg Fantasy Points (PPR)', exclude=['fantasy_points'])
nonppr_corr_bar_chart = viz.corr_bar_chart(all_nonppr, 'avg_fantasy_points', 'Correlation with Avg Fantasy Points (Non-PPR)', exclude=['fantasy_points'])

# correlation of every numeric stat with the target, for each format
ppr_num = all_ppr.select_dtypes(include='number').copy()
nonppr_num = all_nonppr.select_dtypes(include='number').copy()

both = pd.DataFrame({
    'PPR': ppr_num.drop(columns=['avg_fantasy_points', 'fantasy_points']).corrwith(ppr_num['avg_fantasy_points']),
    'Non-PPR': nonppr_num.drop(columns=['avg_fantasy_points', 'fantasy_points']).corrwith(nonppr_num['avg_fantasy_points']),
}).dropna()
both = both[(both.abs() >= 0.3).any(axis=1)].rename_axis('stat').reset_index()
both['difference'] = both['PPR'] - both['Non-PPR']

difference_chart = alt.Chart(both, title='Change in Correlation: PPR − Non-PPR').mark_bar().encode(
    x=alt.X('difference:Q', title='Δ r (PPR − Non-PPR)'),
    y=alt.Y('stat:N', sort='-x', title=None),
    color=alt.condition('datum.difference > 0', alt.value('#2b6cb0'), alt.value('#dd6b20')),
    tooltip=['stat', alt.Tooltip('PPR:Q', format='.2f'), alt.Tooltip('Non-PPR:Q', format='.2f'),
             alt.Tooltip('difference:Q', format='+.2f')],
).properties(width=400, height=alt.Step(18))

(
    (ppr_corr_bar_chart | nonppr_corr_bar_chart)
    & difference_chart
).resolve_scale(
    color='independent'
).properties(
    title='Correlation with Average Fantasy Points per Game (weak correlations, |r| < 0.3, hidden)'
)

alt.VConcatChart(...)

### Which posistions do these statics belong to?

Scoring format clearly changes which positions matter most. 

Football fans may already know which positions produce these stats, but several positions often contribute to the same category, so the chart below shows how each stat's total is split by position.

In [ ]:
TARGET = 'avg_fantasy_points'
EXCLUDE = ['fantasy_points']
POSITIONS = ['QB', 'RB', 'WR', 'TE']
COLORS = ['#2b6cb0', '#38a169', '#dd6b20', '#805ad5']

rows = []
for fmt, df in [('PPR', all_ppr), ('Non-PPR', all_nonppr)]:
    num_cols = df.select_dtypes(include='number').columns.drop([TARGET, *EXCLUDE], errors='ignore')

    # top 10 stats by |r| with the target for this format
    corr = df[num_cols].corrwith(df[TARGET])
    top10 = corr.abs().nlargest(10).index

    # each position's share of each stat's total
    totals = df.groupby('position')[list(top10)].sum().clip(lower=0).reindex(POSITIONS).fillna(0)
    shares = totals / totals.sum()

    for stat in top10:
        start = 0
        for i, pos in enumerate(POSITIONS):
            share = shares.loc[pos, stat]
            rows.append({'format': fmt, 'stat': stat, 'position': pos, 'pos_order': i,
                         'share': share, 'start': start, 'end': start + share,
                         'mid': start + share / 2, 'correlation': corr[stat]})
            start += share

stat_ownership = pd.DataFrame(rows)

y_sort = alt.EncodingSortField('correlation', op='max', order='descending')

bars = alt.Chart().mark_bar().encode(
    x=alt.X('start:Q', axis=alt.Axis(format='%'), title='Share of stat total', scale=alt.Scale(domain=[0, 1])),
    x2='end:Q',
    y=alt.Y('stat:N', title=None, sort=y_sort),
    color=alt.Color('position:N', scale=alt.Scale(domain=POSITIONS, range=COLORS), title='Position'),
    tooltip=['format', 'stat', 'position', alt.Tooltip('share:Q', format='.0%'),
             alt.Tooltip('correlation:Q', format='.2f', title='r with target')],
)

labels = alt.Chart().mark_text(color='white', fontSize=10, fontWeight='bold').encode(
    x='mid:Q',
    y=alt.Y('stat:N', sort=y_sort),
    text=alt.Text('share:Q', format='.0%'),
).transform_filter(
    'datum.share >= 0.06'            # skip labels on slivers too narrow to fit text
)

alt.layer(bars, labels, data=stat_ownership).properties(
    width=320, height=450
).facet(
    column=alt.Column('format:N', title=None, sort=['PPR', 'Non-PPR'], header=alt.Header(labelFontSize=13)),
).resolve_scale(
    y='independent'
).properties(
    title='Top 10 Stats by Correlation: Which Position Produces Them'
)

alt.FacetChart(...)

### **Key Finding #1** 

### The Most Important Statistics by Scoring Format

**PPR**: PPR players should focus on targeting the Wide Receiver posistion in drafts, as they own the largest share of 9/10 statistics with the highest correlations to fantasy points per game. 

**Non-PPR**: Non-PPR players should focus on targeting the Quarterback posistion in drafts, as they own the largest share of 9/10 statistics with the highest correlations to fantasy points per game. 



Now that we know which posistion is the most important, we need to rank all posisitions in order of performance. This will provide key strategic insights into how players of each scoring type should prioritize posistion groups. 

## Position-Specific Drivers

In [ ]:
alt.data_transformers.disable_max_rows()

POSITIONS = list(viz.POSITION_COLORS)
COLOR_SCALE = viz.POSITION_COLOR_SCALE
VALUE = 'avg_fantasy_points'
SEASONS = None   # e.g. [2024] or list(range(2021, 2026)); None = all seasons

for label, df in [('PPR', all_ppr), ('Non-PPR', all_nonppr)]:
    x_title = f'Avg Fantasy Points per Game ({label})'

    # one row per player-season: average fantasy points per game
    data = (
        df
        .groupby(['player', 'position', 'season'], as_index=False)[VALUE]
        .mean()
    )[['position', 'season', VALUE]].dropna()
    if SEASONS is not None:
        data = data[data['season'].isin(SEASONS)]

    # histogram: bars + count labels, one panel per position
    hist_base = alt.Chart().encode(
        x=alt.X(f'{VALUE}:Q', bin=alt.Bin(maxbins=30), title=x_title),
        y=alt.Y('count():Q', title='Player-Seasons'),
    )

    hist_bars = hist_base.mark_bar().encode(color=alt.Color('position:N', scale=COLOR_SCALE, legend=None))
    hist_labels = hist_base.mark_text(dy=-5, fontSize=9).encode(text='count():Q')

    hist = alt.layer(hist_bars, hist_labels, data=data).properties(width=380, height=120).facet(
        row=alt.Row('position:N', sort=POSITIONS, title=None)
    ).resolve_scale(y='independent').properties(title=f'Distribution of Player Averages ({label})')

    # box plot: boxes + labels for the 25th percentile, median and 75th percentile
    box_base = alt.Chart(data).encode(y=alt.Y('position:N', sort=POSITIONS, title=None))

    box_marks = box_base.mark_boxplot(extent=1.5, size=28, median={'color': 'white'},
                                      outliers={'size': 8, 'opacity': 0.3}).encode(
        x=alt.X(f'{VALUE}:Q', title=x_title),
        color=alt.Color('position:N', scale=COLOR_SCALE, legend=None),
    )

    stats = box_base.transform_aggregate(
        q1=f'q1({VALUE})', median=f'median({VALUE})', q3=f'q3({VALUE})', groupby=['position'],
    )
    median_labels = stats.mark_text(dy=-22, fontSize=10, fontWeight='bold').encode(
        x='median:Q', text=alt.Text('median:Q', format='.1f'))
    q1_labels = stats.mark_text(dy=24, fontSize=9, color='gray').encode(
        x='q1:Q', text=alt.Text('q1:Q', format='.1f'))
    q3_labels = stats.mark_text(dy=24, fontSize=9, color='gray').encode(
        x='q3:Q', text=alt.Text('q3:Q', format='.1f'))

    box = (box_marks + median_labels + q1_labels + q3_labels).properties(
        width=380, height=520, title=f'Spread of Player Averages ({label})')

    display((hist | box).properties(title=f'Average Fantasy Points per Game by Position ({label})'))

alt.HConcatChart(...)

alt.HConcatChart(...)

**Early patterns in positional importance**

**Both scoring formats**
1. **QBs score the most.** Their median points per game is consistently higher than any skill position (RB, WR, TE).
2. **Usable QBs are plentiful.** QB averages follow a roughly normal distribution, so many quarterbacks deliver solid, consistent production. Because a starter is easy to find, drafting a QB early is likely less important than filling other positions.
3. **Good TEs are scarce.** TEs have the lowest averages, and few exceed 6 points per game. That makes most TEs a low-impact pick, and the rare productive ones especially valuable.

**PPR**
- Skill positions score more than in non-PPR, which narrows the gap with QBs. With starting QBs easy to find, **skill-position picks matter most.**

**Non-PPR**
- Skill positions average about 1.5 fewer points per game than in PPR, which widens the gap with QBs. **An elite QB may be more valuable in this format**, though further analysis is needed to confirm it.

Before ranking posistions by importance, we need to see if they are consistent over time. 

Are these findings tried and true patterns, or outlier driven?

In [ ]:
alt.data_transformers.disable_max_rows()

POSITIONS = list(viz.POSITION_COLORS)
VALUE = 'avg_fantasy_points'
STEP = 28        # height of each season's row
OVERLAP = 1.2    # how far each ridge rises into the row above

# one row per player-season, for each format
formats = {
    label: df.groupby(['player', 'position', 'season'], as_index=False)[VALUE].mean()
    for label, df in [('PPR', all_ppr), ('Non-PPR', all_nonppr)]
}

for pos in POSITIONS:
    pos_data = {label: ps.loc[ps['position'] == pos, ['season', VALUE]].dropna() for label, ps in formats.items()}

    # shared x-axis across both formats so they're directly comparable
    x_max = float(max(d[VALUE].quantile(0.99) for d in pos_data.values()))
    x_scale = alt.Scale(domain=[0, x_max])

    panels = []
    for label, data in pos_data.items():
        # distribution ridges
        ridges = alt.Chart().transform_filter(
            alt.datum[VALUE] <= x_max
        ).transform_density(
            VALUE, groupby=['season'], as_=[VALUE, 'density'], extent=[0, x_max], steps=100,
        ).transform_joinaggregate(
            max_density='max(density)', groupby=['season']
        ).transform_calculate(
            scaled='datum.density / datum.max_density'   # every ridge peaks at the same height
        ).mark_area(
            interpolate='monotone', fillOpacity=0.75, stroke='white', strokeWidth=0.8,
            color=viz.POSITION_COLORS[pos],
        ).encode(
            x=alt.X(f'{VALUE}:Q', title=f'Avg Fantasy Pts / Game ({label})', scale=x_scale),
            y=alt.Y('scaled:Q', axis=None, scale=alt.Scale(range=[STEP, -STEP * OVERLAP])),
            tooltip=['season'],
        )

        # median marker + label per season
        medians = alt.Chart().transform_aggregate(median=f'median({VALUE})', groupby=['season'])

        median_rule = medians.mark_rule(color='black', strokeWidth=1.5, strokeDash=[3, 2]).encode(
            x=alt.X('median:Q', scale=x_scale),
            y=alt.value(STEP), y2=alt.value(-STEP * 0.6),
            tooltip=['season', alt.Tooltip('median:Q', format='.1f', title='Median pts/game')],
        )
        median_label = medians.mark_text(align='left', dx=4, dy=-4, fontSize=10, fontWeight='bold').encode(
            x=alt.X('median:Q', scale=x_scale),
            y=alt.value(STEP),
            text=alt.Text('median:Q', format='.1f'),
        )

        panels.append(
            alt.layer(ridges, median_rule, median_label, data=data).properties(
                width=380, height=STEP,
            ).facet(
                row=alt.Row('season:O', title=None, header=alt.Header(labelAngle=0, labelAlign='left', labelFontSize=11)),
                spacing=0,
            ).properties(title=label, bounds='flush')
        )

    display(
        alt.hconcat(*panels, spacing=30).properties(
            title=f'{pos}: Distribution of Avg Fantasy Points per Game by Season'
        ).configure_view(stroke=None)
    )

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

### Key Finding #2: 

### Positional Importance Differs by Scoring Format 

**Relevant to Both Formats**
- **RBs and WRs outscore TEs.** TEs are tied to more of the high-correlation stats as their offensive responsibilities similar to Wide Receivers , but RBs score more points per game.
- **Productive RBs are easier to find than productive TEs.** More RBs make consistent contributions, so drafting a useful RB is more likely than drafting a useful TE.
- **QBs provide consistent scoring, but are plentiful.** They have the highest average points per game, but the distribution indicates that there are no shortage of safe options.

**PPR**
- **WRs produce most of the stats that drive scoring.** They account for the largest share of the stats most correlated with points per game.
- **WRs and RBs lead the skill positions in scoring.** Their average points per game are roughly tied.
- **WRs have a higher floor than RBs.** Their 25th-percentile scoring is higher, so even lower-end WRs contribute more reliably.

**Non-PPR**
- **QBs dominate the stats that drive scoring.** They hold the largest share of 9 of the 10 stats most correlated with points per game, making QB much more important in this format. 

While this context is helpful, we want to go one level deeper and suggest a strategy around which posistions to target first by scoring format, ranked by urgency. 

**Methodology**
- **Data:** NFL regular-season weekly stats, 2018–2025 (2026 excluded as a partial season), limited to QB, RB, WR and TE players with at least 6 games in a season.
- **Per-game scoring:** each player's average fantasy points per game for the season, computed separately for PPR and non-PPR scoring.
- **Positional rank:** players are ranked within their position each season by points per game (1 = best), then the score at each rank is averaged across seasons. Each point therefore represents a *typical* Nth-best player, not a specific player.
- **Replacement level:** the first non-starter at each position, assuming a 12-team league, every team must start a specific number of players at each posision. EX: If league has two wide receiver slots per team, and there are 12 teams, this would be the 25th ranked receiver.  
- **Value over replacement:** each rank's points per game minus its position's replacement level. Higher values and steeper curves mean bigger gains from drafting that position early.

## Positional Draft Strategy

In [ ]:
POSITIONS = list(viz.POSITION_COLORS)
COLOR_SCALE = viz.POSITION_COLOR_SCALE
STARTERS = {'QB': 12, 'RB': 24, 'WR': 24, 'TE': 12}   # Required starters for a 12-team league: 1 QB, 2 RB, 2 WR, 1 TE
MAX_RANK = 40   # Plot the top N positional ranks
MIN_GAMES = 6   # Remove players with fewer than this many games

# positions sharing a replacement rank (e.g. QB13 and TE13) get stacked label slots so they don't overlap
LABEL_SLOT, taken = {}, {}
for pos in POSITIONS:
    LABEL_SLOT[pos] = taken.get(STARTERS[pos], 0)
    taken[STARTERS[pos]] = LABEL_SLOT[pos] + 1

rows = []
for label, df in [('PPR', all_ppr), ('Non-PPR', all_nonppr)]:
    weekly = df[
        df['position'].isin(POSITIONS)
    ]

    # aggregate week -> player-season
    ps = (
        weekly.groupby(['player', 'position', 'season'], as_index=False)
        .agg(games=('week', 'nunique'), avg_fantasy_points=('fantasy_points', 'mean'))
    )
    ps = ps[ps['games'] >= MIN_GAMES]

    # rank within position and season (1 = best)
    ps['pos_rank'] = ps.groupby(['season', 'position'])['avg_fantasy_points'].rank(ascending=False, method='first')

    # average points at each positional rank across seasons
    curve = ps.groupby(['position', 'pos_rank'], as_index=False)['avg_fantasy_points'].mean()

    # replacement level = first player past the starters
    repl = curve[curve['pos_rank'] == curve['position'].map(STARTERS) + 1].set_index('position')['avg_fantasy_points']
    curve['above_replacement'] = curve['avg_fantasy_points'] - curve['position'].map(repl)

    # flag and label the replacement-level rank for each position (e.g. "RB25: 12.3 pts/g")
    curve['is_repl'] = curve['pos_rank'] == curve['position'].map(STARTERS) + 1
    curve['repl_label'] = (curve['position'] + curve['pos_rank'].astype(int).astype(str) + ': '
                           + curve['avg_fantasy_points'].map('{:.1f}'.format) + ' pts/g')
    curve['label_slot'] = curve['position'].map(LABEL_SLOT)

    curve['format'] = label
    rows.append(curve[curve['pos_rank'] <= MAX_RANK])

curves = pd.concat(rows, ignore_index=True)

base = alt.Chart().encode(
    x=alt.X('pos_rank:Q', title='Positional rank (1 = best at position)', scale=alt.Scale(domain=[1, MAX_RANK])),
    y=alt.Y('above_replacement:Q', title='Pts / game above replacement'),
    color=alt.Color('position:N', scale=COLOR_SCALE, sort=POSITIONS, title='Position'),
)

lines = base.mark_line(point=alt.OverlayMarkDef(size=20)).encode(
    tooltip=['format', 'position', alt.Tooltip('pos_rank:Q', title='Rank'),
             alt.Tooltip('avg_fantasy_points:Q', format='.1f', title='Pts / game'),
             alt.Tooltip('above_replacement:Q', format='+.1f', title='Above replacement')],
)

# replacement level: diamond marker + label (stacked where positions share a rank)
repl_points = base.transform_filter(alt.datum.is_repl).mark_point(
    shape='diamond', size=140, filled=True, stroke='black', strokeWidth=1, opacity=1,
).encode(
    tooltip=['format', 'position', alt.Tooltip('pos_rank:Q', title='Replacement rank'),
             alt.Tooltip('avg_fantasy_points:Q', format='.1f', title='Replacement pts / game')],
)
repl_labels = [
    base.transform_filter((alt.datum.is_repl) & (alt.datum.label_slot == slot)).mark_text(
        align='left', dx=8, dy=-14 + 16 * slot, fontSize=10, fontWeight='bold',
    ).encode(text='repl_label:N')
    for slot in sorted(set(LABEL_SLOT.values()))
]

zero = alt.Chart().mark_rule(color='gray', strokeDash=[4, 4]).encode(y=alt.datum(0))

alt.layer(zero, lines, repl_points, *repl_labels, data=curves).properties(width=500, height=450).facet(
    column=alt.Column('format:N', sort=['PPR', 'Non-PPR'], title=None, header=alt.Header(labelFontSize=13)),
).properties(title='Positional Drop-off: Points per Game Above Replacement (◆ = replacement level)')

alt.FacetChart(...)

### Key Finding #3: Ranking Positions by Draft Priority

Positions are ranked by how much the best player at each position outscores a replacement level player in points per game. A bigger gap means waiting on that position costs more.

**PPR**
1. **Running Back:** +11.7 pts/game
   - WRs produce the largest share of high-correlation stats, but RBs drop off more steeply, so elite RBs are harder to replace and should be drafted first.
2. **Wide Receiver:** +10.1 pts/game
3. **Tight End:** +8.3 pts/game
4. **Quarterback:** +7.6 pts/game

**Non-PPR**
1. **Running Back:** +10.1 pts/game
   - QBs produce the largest share of high-correlation stats, but RBs drop off more steeply, so elite RBs remain the top priority.
2. **Quarterback:** +7.6 pts/game
3. **Wide Receiver:** +6.7 pts/game
4. **Tight End:** +5.5 pts/game

This provides a strategic framework for fantasy players to approach how they should select their team. 

While the blueprint is helpful, it doesn't help us to identify which players to take. Given the ethical concern we raised above, we will refrain from providing specifc player recommendations and instead provide analysis to highlight general statstics that players use to choose optimal players at each posistion. 

## Player Selection by Posistion 

### Correlations

Only correlations with |r| ≥ 0.3 are shown, to focus on the most impactful features.

Stats that are part of the scoring formula itself, such as receiving yards and touchdowns, are excluded. They correlate with fantasy points by definition, so including them would show what *produces* points rather than what *drives* a player to score them.

In [ ]:
POSITIONS = list(viz.POSITION_COLORS)
MIN_CORR = 0.3

# columns that are IDs, time, or the target itself in another form
EXCLUDE = [
    # bookkeeping / target variants
    'avg_fantasy_points', 'fantasy_points_ppr', 'total_games_played', 'season', 'week',

    # touchdowns
    'passing_tds', 'rushing_tds', 'receiving_tds',

    # yards that score points (and their per-game versions)
    'passing_yards', 'rushing_yards', 'receiving_yards',
    'pass_yards_per_game', 'rush_yards_per_game', 'rec_yards_per_game',

    # receptions (worth 1 point each in PPR)
    'receptions',

    # 2-point conversions
    'passing_2pt_conversions', 'rushing_2pt_conversions', 'receiving_2pt_conversions',

]

pairs = []
for pos in POSITIONS:
    ppr_chart = viz.targeted_heatmap(
        all_ppr[all_ppr['position'] == pos], ['fantasy_points'],
        title='PPR', exclude=EXCLUDE, min_corr=MIN_CORR,
    )
    nonppr_chart = viz.targeted_heatmap(
        all_nonppr[all_nonppr['position'] == pos], ['fantasy_points'],
        title='Non-PPR', exclude=EXCLUDE, min_corr=MIN_CORR,
    )
    pairs.append((ppr_chart | nonppr_chart).properties(title=pos))

alt.vconcat(
    alt.hconcat(pairs[0], pairs[1], spacing=60),
    alt.hconcat(pairs[2], pairs[3], spacing=60),
    spacing=40,
).properties(
    title=f'Stats Correlated with Fantasy Points by Position (|r| ≥ {MIN_CORR})'
)

alt.VConcatChart(...)

### Quarterback 



In [ ]:
# columns lists
all_positions_cols = [
    'player_id',
    'player_name',
    'player',
    'position',
    'position_group',
    'headshot_url',
    'season',
    'week',
    'season_type',
    'game_id',
    'team',
    'opponent_team',
    'data_source',
    'games_played_unit'
]

qb_cols = [
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'sacks_suffered',
    'sack_yards_lost',
    'sack_fumbles',
    'sack_fumbles_lost',
    'passing_air_yards',
    'passing_yards_after_catch',
    'passing_first_downs',
    'passing_epa',
    'passing_cpoe',
    'passing_2pt_conversions',
    'pacr',
    'passing_10',
    'passing_16',
    'passing_20',
    'passing_40',
    'carries',
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles',
    'rushing_fumbles_lost',
    'rushing_first_downs',
    'rushing_epa',
    'rushing_2pt_conversions',
    'rushing_10',
    'rushing_12',
    'rushing_20',
    'rushing_40'
]

rb_cols = [
    'carries',
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles',
    'rushing_fumbles_lost',
    'rushing_first_downs',
    'rushing_epa',
    'rushing_2pt_conversions',
    'rushing_10',
    'rushing_12',
    'rushing_20',
    'rushing_40',
    'receptions',
    'targets',
    'receiving_yards',
    'receiving_tds',
    'receiving_fumbles',
    'receiving_fumbles_lost',
    'receiving_air_yards',
    'receiving_yards_after_catch',
    'receiving_first_downs',
    'receiving_epa',
    'receiving_2pt_conversions',
    'receiving_10',
    'receiving_16',
    'receiving_20',
    'receiving_40',
    'racr',
    'target_share',
    'air_yards_share'
]

wr_cols = [
    'receptions',
    'targets',
    'receiving_yards',
    'receiving_tds',
    'receiving_fumbles',
    'receiving_fumbles_lost',
    'receiving_air_yards',
    'receiving_yards_after_catch',
    'receiving_first_downs',
    'receiving_epa',
    'receiving_2pt_conversions',
    'receiving_10',
    'receiving_16',
    'receiving_20',
    'receiving_40',
    'racr',
    'target_share',
    'air_yards_share'
]

def_cols = [
    'def_tackles_solo',
    'def_tackles_with_assist',
    'def_tackle_assists',
    'def_tackles_for_loss',
    'def_tackles_for_loss_yards',
    'def_fumbles_forced',
    'def_sacks',
    'def_sack_yards',
    'def_qb_hits',
    'def_interceptions',
    'def_interception_yards',
    'def_pass_defended',
    'def_tds',
    'def_fumbles',
    'def_safeties',
    'def_punt_blocks',
    'def_pat_blocks',
    'def_fg_blocks',
    'def_2pt_atts',
    'def_2pt_made',
    'misc_yards',
    'fumble_recovery_own',
    'fumble_recovery_yards_own',
    'fumble_recovery_opp',
    'fumble_recovery_yards_opp',
    'fumble_recovery_tds'
]

In [ ]:
qb_season_agg = pd.read_csv("data/qb_season_agg.csv")

In [ ]:
# create lists for filters
season_list = [2018,2019,2020,2021,2022,2023,2024,2025,2026]
player_list = qb_season_agg['player'].unique().tolist()
season_filter = alt.param(
    name = 'selected_season',
    value = 2024,
    bind = alt.binding_select(
        options = season_list,
        name = 'Season'
    )
)
bar_chart_passing_yards = (alt.Chart(qb_season_agg).mark_bar().encode(
    x=alt.X('player:N', title='Player Name'),
    y=alt.Y('passing_yards:Q', title='Passing Yards'),
    color=alt.Color('season:N', title='Season'),
    tooltip=['player', 
             'season', 
             'passing_yards', 
             'passing_tds', 
             'passing_interceptions', 
             'completion_percentage', 
             'passing_yards_per_game']
).add_params(season_filter)
.transform_filter(alt.datum.season == season_filter)
).properties(
    title='Passing Yards by Player and Season'
).interactive()

mean_passing_yards_line = (
    alt.Chart(qb_season_agg).transform_filter(alt.datum.season == season_filter)
    .transform_calculate(
        statistic="'Mean Passing Yards'"
    )
    .mark_rule(
        strokeWidth=2
    )
    .encode(
        y=alt.Y(
            'mean(passing_yards):Q',
            title='Mean Passing Yards'
        ),
    color=alt.Color(
        'statistic:N',
        scale=alt.Scale(
            domain = ['Mean Passing Yards'],
            range = ['red']
        )
    )
)
)

chart = (
    bar_chart_passing_yards + mean_passing_yards_line).resolve_scale(color='independent')

chart


alt.LayerChart(...)

In [ ]:
# boxplot for passing yards by season
boxplot_passing_yards = (
    alt.Chart(qb_season_agg).mark_boxplot().encode(
        x=alt.X('season:N', title='Season'),
        y=alt.Y('passing_yards:Q', title='Passing Yards'),
        color=alt.Color('season:N', title='Season')
    )
    .properties(
        width = 750,
        height = 100,
        title = 'Passing Yards by Season'
    )
)

boxplot_passing_yards

alt.Chart(...)

In [ ]:
# heatmap of correlations
corr = qb_season_agg.select_dtypes(include='number').corr().reset_index().melt(id_vars='index')
corr.columns = ['var1', 'var2', 'correlation']

qb_heatmap = alt.Chart(corr, title="Correlation Heatmap — QB Stats").mark_rect().encode(
    x=alt.X('var1:N', title=None, sort=None),
    y=alt.Y('var2:N', title=None, sort=None),
    color=alt.Color('correlation:Q', scale=alt.Scale(scheme='magma', domain=[-1, 1])),
    tooltip=['var1', 'var2', alt.Tooltip('correlation:Q', format='.2f')]
).properties(width=500, height=500)

qb_heatmap

alt.Chart(...)

In [ ]:
qb_season_agg.columns

Index(['player', 'position', 'season', 'data_source', 'completions',
       'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions',
       'carries', 'rushing_yards', 'rushing_tds', 'rushing_2pt_conversions',
       'sacks_suffered', 'sack_yards_lost', 'sack_fumbles',
       'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
       'passing_first_downs', 'passing_epa', 'passing_cpoe', 'pacr',
       'total_games_played', 'completion_percentage', 'passing_yards_per_game',
       'fantasy_points', 'avg_fantasy_points'],
      dtype='object')

In [ ]:
qb_dashboard = viz.build_qb_dashboard(qb_season_agg[qb_season_agg['total_games_played'] >= 3])
out_path = "qb_dashboard.html"
qb_dashboard.save(out_path)
print(f"Saved dashboard to {out_path}")
qb_dashboard


Saved dashboard to qb_dashboard.html


alt.VConcatChart(...)

### Wide Receiver 

#### Air Yards vs Target Share 

#### Target Share vs Receiving Yards 

### Tight End 

#### Air Yards vs Target Share 

#### Target Share vs Receiving Yards 

### Running Back 